# 🛰️ Nexus_SIH: Sentinel-2 Multi-Spectral SRGAN Fine-Tuning in Google Colab

Welcome to the **Nexus SIH** Super-Resolution training pipeline!

### What this notebook does:
1. **GPU Acceleration**: Verifies NVIDIA T4 GPU runtime.
2. **Repository & Dependencies**: Clones `Nexus_SIH` and installs satellite raster processing tools.
3. **Data Preparation**: Slices raw Sentinel-2 4-band multi-spectral images (R, G, B, NIR) into $64\times 64$ Low-Resolution (10m) and $256\times 256$ High-Resolution (2.5m) training patches.
4. **Fine-Tuning**: Trains the **SentinelSRGAN** (Generator + Discriminator) with Spectral Angle Mapper (SAM) loss to preserve NDVI & plant health signatures.
5. **Evaluation & Uncertainty Map**: Measures PSNR, SSIM, SAM, and visualizes the AI's confidence map.
6. **Export**: Saves `generator_best.pth` and `srgan_sentinel2_x4.pth` to Google Drive or local download.

### ⚡ Step 0: Ensure GPU is Enabled
Go to the menu: **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> Click **Save**.

In [ ]:
!nvidia-smi

### 📥 Step 1: Clone Repository and Install Requirements

In [ ]:
# Clone the repository (if not already cloned)
import os
if not os.path.exists('Nexus_SIH'):
    !git clone https://github.com/upamkmr/Nexus_SIH.git

%cd /content/Nexus_SIH

# Install geospatial & ML dependencies
!pip install -q pyyaml tifffile scikit-image rasterio

### ✂️ Step 2: Slice the Data into Training Patches
If you have your own Sentinel-2 `.tif` files, you can upload them to `data/raw/sentinel2/`.
If the folder is empty, the script will automatically generate synthetic 4-band multi-spectral satellite tiles so you can immediately prototype!

In [ ]:
# Slices raw Sentinel-2 multi-spectral images into 64x64 LR and 256x256 HR patches (4x Super-Resolution)
!python scripts/prepare_dataset.py --source data/raw/sentinel2 --output data/processed/train --patch-size 64 --scale 4

### ⚙️ Step 3: Check & Tweak Hyperparameters
Our settings in `ml-service/configs/srgan_config.yaml`:
- `lr_generator`: `0.0001` (1e-4) to avoid forgetting pre-trained features
- `batch_size`: `8` (well within Colab T4 memory limits)
- `epochs`: `25`
- `lambda_sam`: `0.05` (Spectral Angle preservation for NDVI/NIR consistency)

In [ ]:
# Inspect current configuration
!cat ml-service/configs/srgan_config.yaml

### 🚀 Step 4: Run the SRGAN Fine-Tuning Loop
Watch the Generator and Discriminator compete, logging PSNR and Spectral Angle Mapper (SAM) values.

In [ ]:
!python ml-service/training/trainer.py --config ml-service/configs/srgan_config.yaml --epochs 25 --batch-size 8

### 🔍 Step 5: Evaluate the Results & Visualize Uncertainty Map
This runs the trained model on validation patches, calculates PSNR/SSIM/SAM, and generates an **AI Uncertainty Map** showing where the AI is confident vs. where edge reconstruction required higher hallucination correction.

In [ ]:
!python scripts/evaluate_model.py --weights checkpoints/generator_best.pth --data-dir data/processed/val --output-dir data/outputs

### 🖼️ Step 6: Display Before & After Comparison in Colab

In [ ]:
from IPython.display import Image, display
comparison_path = 'data/outputs/evaluation_comparison.png'
if os.path.exists(comparison_path):
    display(Image(filename=comparison_path))
else:
    print('Evaluation image not found.')

### 💾 Step 7: Download Trained Weights to Your Computer
Download the fine-tuned `.pth` file so you can drop it directly into your local `Nexus_SIH/ml-service/checkpoints/` folder!

In [ ]:
from google.colab import files
import os

best_weights = 'checkpoints/generator_best.pth'
if os.path.exists(best_weights):
    files.download(best_weights)
    print('Downloading generator_best.pth...')
else:
    print('Checkpoint not found. Make sure training completed successfully.')